# Xiangqi AI — Google Colab Bronze → Silver

Runner độc lập cho dữ liệu Alpha-Beta: mount Google Drive, clone main, sao lưu nguồn vào Bronze (không ghi đè), replay/validate/deduplicate bằng SilverPipeline trong backend và copy kết quả lên Drive.

Nguồn hiện có: MyDrive/XiangqiAI/datasets. Đầu ra: MyDrive/XiangqiAI/data/01_bronze và 02_silver. Chưa xây dựng Gold, chưa hỗ trợ CCPD. Không sinh thêm ván cờ, không benchmark, không chạy training.

Chạy tuần tự từ trên xuống trên Google Colab CPU. Mặc định chạy 20 dòng đầu để smoke-test; sau khi xem quarantine mới đổi MODE thành full. Bạn phải cấp quyền Google Drive mount trong phiên Colab.

## 1. Mount Drive và cấu hình

Chỉnh SOURCE_FILES nếu cần; các file .checkpoint.json không phải dữ liệu ván cờ. Tài khoản mount phải truy cập được XiangqiAI/datasets.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/XiangqiAI")
SOURCE_DIR = DRIVE_PROJECT / "datasets"
BRONZE_DIR = DRIVE_PROJECT / "data" / "01_bronze" / "alphabeta"
SILVER_DIR = DRIVE_PROJECT / "data" / "02_silver"
REPO_DIR = Path("/content/ML-Advance-Group")
BACKEND_DIR = REPO_DIR / "project" / "backend"
WORK_DIR = Path("/content/xiangqi_silver_runs")

MODE = "sample"  # "sample" (first N games per file) or "full"
SAMPLE_GAMES = 20
SOURCE_FILES = ["alphabeta_depth3_book_topk3_dedup_v2_seed42_max240.jsonl"]
SOURCE_VERSION = "colab-batch-v1"
SOURCE_LICENSE = None  # No assumed license.
BRANCH = "main"

if MODE not in {"sample", "full"} or (MODE == "sample" and SAMPLE_GAMES <= 0):
    raise ValueError("Invalid mode or sample size")
if not SOURCE_FILES or len(SOURCE_FILES) != len(set(SOURCE_FILES)):
    raise ValueError("SOURCE_FILES must be unique and non-empty")
if any(Path(name).name != name or not name.endswith(".jsonl") for name in SOURCE_FILES):
    raise ValueError("Use simple .jsonl filenames, not paths")
if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f"Drive source folder not accessible: {SOURCE_DIR}")
print("Source directory:", SOURCE_DIR)

## 2. Clone source và cài backend

Dùng main hiện tại, không phụ thuộc notebook cũ. Fast-forward khi thư mục clone đã tồn tại; không xóa local changes.

In [ ]:
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCH,
        "https://github.com/Ko4rut/ML-Advance-Group.git", str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

assert (BACKEND_DIR / "src/data/02_silver/data_pipeline/pipeline.py").is_file()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND_DIR)], check=True)
REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Git revision:", REVISION)

## 3. Kiểm tra file nguồn

Tên dataset mặc định đã được đối chiếu với Drive của nhóm. Bạn có thể thêm nhiều .jsonl vào SOURCE_FILES để deduplicate trên toàn bộ batch.

In [ ]:
print("Available JSONL sources:")
for file in sorted(SOURCE_DIR.glob("*.jsonl")):
    print(f"- {file.name} ({file.stat().st_size:,} bytes)")
sources = [SOURCE_DIR / filename for filename in SOURCE_FILES]
for file in sources:
    if not file.is_file() or file.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty JSONL: {file}")
print("Selected:", [file.name for file in sources])

## 4. Ingest bất biến sang Bronze

Copy toàn bộ file JSONL gốc và tạo manifest cạnh mỗi file. Nếu đã tồn tại, so SHA-256; khác hash thì dừng, không ghi đè.

In [ ]:
import hashlib
import json
import shutil
from datetime import datetime, timezone

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
bronze_files = []
for source in sources:
    target = BRONZE_DIR / source.name
    digest = sha256_file(source)
    if target.exists():
        if sha256_file(target) != digest:
            raise ValueError(f"Bronze name collision with different bytes: {target}")
    else:
        shutil.copy2(source, target)
        if sha256_file(target) != digest:
            raise IOError(f"Bronze copy failed checksum: {target}")
    sidecar = target.with_name(target.name + ".manifest.json")
    if sidecar.exists():
        if json.loads(sidecar.read_text(encoding="utf-8")).get("file_sha256") != digest:
            raise ValueError(f"Invalid existing manifest: {sidecar}")
    else:
        manifest = {
            "schema_version": 1, "source": "alphabeta",
            "source_version": SOURCE_VERSION, "source_file": source.name,
            "file_sha256": digest, "ingested_at": datetime.now(timezone.utc).isoformat(),
            "license": SOURCE_LICENSE, "original_path": str(source),
        }
        with sidecar.open("x", encoding="utf-8") as stream:
            json.dump(manifest, stream, ensure_ascii=False, indent=2)
    bronze_files.append(target)
    print("Bronze verified:", source.name, digest[:12])

## 5. Stage sample/full trong /content

Đọc file trên ổ tạm Colab để hạn chế đọc/ghi chậm trên Drive. MODE=sample lấy N dòng không rỗng **đầu tiên** (chỉ smoke-test, không đại diện thống kê). Mỗi run có ID riêng, không ghi đè.

In [ ]:
RUN_ID = "alphabeta_" + MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
local_inputs = []
input_dir = WORK_DIR / RUN_ID / "input"
output_dir = WORK_DIR / RUN_ID / "silver"
input_dir.mkdir(parents=True, exist_ok=False)

for source in bronze_files:
    target = input_dir / source.name
    if MODE == "full":
        shutil.copy2(source, target)
    else:
        count = 0
        with source.open("r", encoding="utf-8") as in_stream:
            with target.open("x", encoding="utf-8") as out_stream:
                for line in in_stream:
                    if not line.strip():
                        continue
                    out_stream.write(line if line.endswith("\n") else line + "\n")
                    count += 1
                    if count >= SAMPLE_GAMES:
                        break
        print("Sampled:", source.name, count)
    if target.stat().st_size == 0:
        raise ValueError(f"Selected source has no records: {source}")
    local_inputs.append(target)
print("Run ID:", RUN_ID)

## 6. Xây dựng Silver bằng GameEngine trong backend

Bộ CLI scripts.build_silver ghi games.jsonl, transitions.jsonl, duplicates.jsonl, quarantine.jsonl. Báo cáo số ván hợp lệ, trùng và bị loại. Các ván nguồn không được sửa.

In [ ]:
command = [
    sys.executable, "-m", "scripts.build_silver",
    *[str(p) for p in local_inputs],
    "--output-dir", str(output_dir),
    "--source-version", SOURCE_VERSION,
]
if SOURCE_LICENSE is not None:
    command.extend(["--license", SOURCE_LICENSE])
result = subprocess.run(command, cwd=BACKEND_DIR, capture_output=True, text=True, check=True)
report = json.loads(result.stdout)
print(json.dumps(report, indent=2))
if report["source_records"] != sum(
    report[k] for k in ("accepted_games", "duplicate_games", "quarantined_records")
):
    raise AssertionError("Inconsistent pipeline totals")
if report["source_records"] == 0:
    raise ValueError("No source records")

## 7. Xem quarantine, xuất report và đồng bộ Silver lên Drive

Chỉ copy khi CLI thành công. Tên run duy nhất; file đầu ra được kiểm tra SHA-256 sau khi copy. Nếu quá trình đồng bộ bị ngắt, thư mục ẩn .uploading còn tồn tại để kiểm tra, không được hiểu là run thành công.

In [ ]:
from collections import Counter

issues = Counter()
with (output_dir / "quarantine.jsonl").open("r", encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            issues[json.loads(line)["error_code"]] += 1
print("Quarantine reasons:", dict(issues))
print("Accepted %:", round(100 * report["accepted_games"] / report["source_records"], 2))
if report["accepted_games"] == 0:
    print("WARNING: zero accepted games, inspect quarantine before training")

metadata = {
    "run_id": RUN_ID, "mode": MODE, "git_revision": REVISION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_files": [{"name": p.name, "sha256": sha256_file(p)} for p in bronze_files],
    "metrics": report, "quarantine_error_codes": dict(issues),
}
(output_dir / "report.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

SILVER_DIR.mkdir(parents=True, exist_ok=True)
final_dir = SILVER_DIR / RUN_ID
staging_dir = SILVER_DIR / ("." + RUN_ID + ".uploading")
if final_dir.exists() or staging_dir.exists():
    raise FileExistsError("Run already exists on Drive; choose a new RUN_ID")
shutil.copytree(output_dir, staging_dir)
for source in output_dir.iterdir():
    if sha256_file(source) != sha256_file(staging_dir / source.name):
        raise IOError(f"Silver Drive checksum failed: {source.name}")
staging_dir.rename(final_dir)
print("Silver saved:", final_dir)
print("Output files:", [p.name for p in sorted(final_dir.iterdir())])

## Kết quả và bước tiếp theo

Sau khi thử sample, đọc report.json và quarantine.jsonl trong thư mục Silver. Nếu dữ liệu hợp lệ, chuyển MODE=full rồi chạy lại từ cell 5; Bronze được giữ nguyên. Gold hiện chưa chạy. Notebook cần runtime Colab có mount Drive: chỉ có thể thực thi phần mount/đồng bộ trên tài khoản người chạy, không phải từ GitHub.